In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import GPT2Config, GPT2LMHeadModel
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

df = pd.read_csv('llmtrace_news.csv')
texts = df['text'].dropna().tolist()
print(f"Текстов: {len(texts)}")

In [ ]:
# 111111111111111111111111111
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")
tokenizer.pad_token = tokenizer.eos_token
vocab_size = tokenizer.vocab_size

# Собираем токены с частотами
token_counts = Counter()
for text in texts:
    token_counts.update(tokenizer.encode(text))

total = sum(token_counts.values())
freqs = {t: c/total for t, c in token_counts.items()}
threshold = 1e-5  # подбираем под соотношение ~100

# Subsampling
tokens_list = []
for text in texts:
    for t in tokenizer.encode(text):
        p_keep = min(1.0, np.sqrt(threshold / freqs[t]))
        if np.random.random() < p_keep:
            tokens_list.append(t)
# 111111111111111111111111111

In [ ]:
# 2222222222222222222222222222
with open('corpus.txt', 'w', encoding='utf-8') as f:
    for text in texts:
        f.write(text + '\n')

# Обучаем BPE через SentencePiece
spm.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='bpe5k',
    vocab_size=5000,
    model_type='bpe',
    character_coverage=1.0
)

sp = spm.SentencePieceProcessor()
sp.load('bpe5k.model')
vocab_size = sp.get_piece_size()

tokens_list = []
for text in texts:
    tokens_list.extend(sp.encode(text))
# 2222222222222222222222222222

In [ ]:
#33333333333333333333333333333333
with open('corpus.txt', 'w', encoding='utf-8') as f:
    for text in texts:
        f.write(text + '\n')

spm.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='unigram30k',
    vocab_size=30000,
    model_type='unigram',
    character_coverage=1.0
)

sp = spm.SentencePieceProcessor()
sp.load('unigram30k.model')
vocab_size = sp.get_piece_size()

tokens_list = []
for text in texts:
    tokens_list.extend(sp.encode(text))
#33333333333333333333333333333333

In [ ]:
#444444444444444444444444444
with open('corpus.txt', 'w', encoding='utf-8') as f:
    for text in texts:
        f.write(text + '\n')

spm.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='unigram10k',
    vocab_size=10000,
    model_type='unigram',
    character_coverage=1.0
)

sp = spm.SentencePieceProcessor()
sp.load('unigram10k.model')
vocab_size = sp.get_piece_size()

tokens_list = []
for text in texts:
    tokens_list.extend(sp.encode(text))
#444444444444444444444444444

In [ ]:
#555555555555555555555555555
with open('corpus.txt', 'w', encoding='utf-8') as f:
    for text in texts:
        f.write(text + '\n')

spm.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='unigram5k_reg',
    vocab_size=5000,
    model_type='unigram',
    character_coverage=1.0,
    shrinking_factor=0.9,
    unigram_piece_sampling='latin'
)

sp = spm.SentencePieceProcessor()
sp.load('unigram5k_reg.model')
vocab_size = sp.get_piece_size()

tokens_list = []
for text in texts:
    tokens_list.extend(sp.encode(text))
#555555555555555555555555555

In [ ]:
block_size = 128
batch_size = 32

all_tokens = torch.tensor(tokens_list, dtype=torch.long)
n_blocks = len(all_tokens) // (block_size + 1)
X = all_tokens[:n_blocks * (block_size + 1)].view(-1, block_size + 1)
inputs, targets = X[:, :-1], X[:, 1:]

dataset = TensorDataset(inputs, targets)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Блоков: {n_blocks}, Vocabulary: {vocab_size}")

In [ ]:
def train(name, loader, vocab_size, lr, epochs=10):
    config = GPT2Config(vocab_size=vocab_size, n_positions=128, n_embd=256, n_layer=4, n_head=4)
    model = GPT2LMHeadModel(config).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr) if 'sgd' in name else torch.optim.Adam(model.parameters(), lr=lr)

    history = []
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        batches = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = model(xb, labels=yb).loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
            batches += 1
        history.append(epoch_loss / batches)
        print(f"{name} epoch {epoch+1}: loss={history[-1]:.4f}")
    return history

print("=== SGD ===")
loss_sgd = train("sgd", loader, vocab_size, lr=0.5)

print("\n=== Adam ===")
loss_adam = train("adam", loader, vocab_size, lr=0.001)

plt.plot(loss_sgd, 's-', label='SGD')
plt.plot(loss_adam, 's-', label='Adam')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.title(f'Experiment: #???')
plt.show()

print(f"Final SGD: {loss_sgd[-1]:.4f}, Final Adam: {loss_adam[-1]:.4f}, Delta: {loss_sgd[-1] - loss_adam[-1]:.4f}")